<a href="https://colab.research.google.com/github/arjunsunar748/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjunsunar748/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring.**

I'm picking this lane because it maps cleanly onto a real bottleneck: a content team has far more pages than
review hours, and right now the choice of "what do we open first" is made informally. The starter pipeline in
this repo already runs this exact question end-to-end on the anonymized starter data and reports a real,
checked benchmark (baseline rule Precision@50 ≈ 0.24 vs. random forest Precision@50 ≈ 0.74 — see
`docs/ml-intern-dataset-and-lane-guide.md`, section 5). That gives me a concrete, honest target to try to beat
(or fail to beat and learn from) instead of starting from nothing. It also produces the artifact I find most
useful — a ranked queue with reason codes a human can actually inspect — rather than a black-box score.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Lane: Refresh / Content Opportunity Scoring")

Lane: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one row = one (pseudonymized) content item — a page — described by its trailing
90-day search and engagement metrics. Not a client, not a day: a page.

**Research question:** Given a content team's limited review capacity, which pages should an editor open
*first* this cycle — for refresh, expansion, protection, pruning, or monitoring?

**Decision improved:** the order of a fixed-size review queue. Today that ordering is informal (whoever
someone happens to remember, or a flat alphabetical/CMS list). The output of this work is a ranked list, not
a yes/no verdict on any single page.

**Who acts, and how:** a content editor or SEO strategist with a fixed weekly review budget (say, the top 20
or 50 pages) opens the queue, reads the reason code attached to each page (e.g. "declining with demand,"
"page-one decay risk"), and decides whether to refresh the content, expand it, protect it as-is, prune it, or
just keep monitoring it. The output has to be inspectable — a bare score with no reason code isn't actionable.

**Cost of a wrong call, and why it's asymmetric:**
- *False positive* (a page ranked high that didn't actually need attention): wastes a scarce editor-hour that
  could have gone to a page that really was declining. With a fixed weekly budget, every wasted slot has an
  opportunity cost, not just a wasted-effort cost.
- *False negative* (a real, high-value decline that never makes the list): the page keeps losing visibility
  silently, and — because impression volume is concentrated (see Section 3) — missing a *high-traffic*
  decline is far more expensive than missing a low-traffic one. This is why a ranking/precision@K framing
  fits better than a plain balanced-accuracy classifier: the top of the list matters much more than the
  middle.

**Why data/ML helps at all:** a single-column rule ("if trend is down, flag it") is easy to write but leaves
signal on the table — declining pages are common (see Section 3), so a rule that fires on that alone floods
the queue with more candidates than any team can review, without ranking within that set. The real priority
signal is a tangled combination of demand, position, freshness, word count, and engagement that shifts by
content type and client — exactly the case the `framing-ml-problems` skill flags as where ML earns its keep
over a plain if-statement. The starter benchmark already shows a learned ranking finding real lift over the
hand-written rule on this same data.


In [21]:

# No numbers needed for this section either - decision/action/cost is a framing exercise.
# The scale numbers that justify "a plain rule floods the queue" are computed in Section 3.
print("Decision: order of a fixed-size weekly review queue")
print("Actor: content editor / SEO strategist")
print("Action: refresh, expand, protect, prune, or monitor the page")

Decision: order of a fixed-size weekly review queue
Actor: content editor / SEO strategist
Action: refresh, expand, protect, prune, or monitor the page


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Loading `data/raw/content_refresh_anonymized.csv` (30,000 pages x 44 columns, 32 clients) and checking the
scale of the review-queue problem directly, using the gotchas from `skills/flyrank/flyrank-data/SKILL.md`
(rate columns are already x100 percentages, `avg_position == 0` means "no data," ids are join keys only).


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

n_total = len(df)
n_clients = df["client_id"].nunique()
print(f"Total pages: {n_total:,} across {n_clients} pseudonymized clients")

# Pages that are visible (impressions_90d >= 500) AND declining (trend_direction == "down")
# — this uses the starter's own "declining_with_demand" style thresholds, not a made-up cutoff.
visible = df[df["impressions_90d"] >= 500]
visible_declining = visible[visible["trend_direction"] == "down"]

print(f"Visible pages (impressions_90d >= 500): {len(visible):,} "
      f"({len(visible) / n_total * 100:.1f}% of all pages)")
print(f"Of those, declining right now: {len(visible_declining):,} "
      f"({len(visible_declining) / len(visible) * 100:.1f}% of visible pages)")

# Where is the impression volume sitting? If it's concentrated in declining pages,
# a wrong (missed) call is expensive, not just annoying.
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
share_of_impressions = declining_with_demand["impressions_90d"].sum() / df["impressions_90d"].sum() * 100
print(f"Share of all trailing-90d impression volume sitting in declining-with-demand pages: "
      f"{share_of_impressions:.1f}%")



FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can say:**
- *Observed / measured:* on this anonymized 30k-page starter slice, roughly 60% of visible pages
  (impressions_90d >= 500) are currently trending down, and over half of all trailing-90-day impression
  volume sits inside that declining group. That's a measurement, not a prediction.
- *Directional:* signals like position tier, freshness, and engagement rate appear associated with which
  pages are declining or under-capturing clicks — a pattern worth testing further, not a proven cause.
- *Decision-support:* the end output is a ranked queue with reason codes meant to help a human reviewer spend
  limited time better. It is a prioritization aid, not an automated fix.

**What this work will never claim:**
- That a refresh *causes* a recovery. Correlational, observational data like this can't establish causation —
  that needs an actual experiment (e.g. A/B refreshing a matched set of pages), which is out of scope here.
- That I've reverse-engineered or predicted anything about Google's ranking algorithm. I'm working with
  FlyRank's own observed search/engagement signals, not Google internals.
- That a high-priority score guarantees a page is actually broken, or that a low score means it's fine — this
  is a triage aid for a scarce-time decision, not a verdict.

**A specific trap I'm flagging now, before Week 2:** the starter dataset's `is_declining_label` is defined as
`trend_direction == "down"` — a bucket computed from the *current* window, not an outcome observed in a
*future* window (see `docs/ml-intern-dataset-and-lane-guide.md`, section 5). If I keep using that proxy
as-is, any model I build is partly learning to reproduce a rule, not to discover something new. My data
contract in Week 3 needs to either defend using this proxy explicitly and honestly, or move toward a
future-window label (e.g. prior 90 days of features -> decline over the next 30 days) before claiming
predictive rather than descriptive results.


In [ ]:

# No additional computation needed here - this section is about the honest scope of the claims,
# not new numbers. The proxy-label caveat above references the label definition from
# docs/ml-intern-dataset-and-lane-guide.md section 5, already read as part of framing-ml-problems.
print("Claims: observed / directional / decision-support only. No causal or Google-algorithm claims.")

